## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from sem_rag.config.settings import *

In [ ]:
import os
import json
import re
from matplotlib import pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd
import torch
from IPython.display import display, FileLink

In [ ]:
prova = np.load(TestSem.CROPS / "111_2d3ed1a61471caa941876befb9b70328fb40bf42c66ba80d5d4817b43b556d41_512.npy")
prova

In [ ]:
provadue = torch.load(TestSem.EMBEDDINGS_DINO / "111.pt")
provadue.size()

In [ ]:
with open(TestSem.JSON / "111_SEMMetaData.json") as f:
    provatre = json.load(f)

len(provatre)

In [ ]:
provatre

In [ ]:
plt.imshow(prova, cmap='gray')
plt.show()

In [ ]:
all_data = []

# Loop through all files in the directory
for filename in os.listdir(TestSem.JSON):
    if filename.endswith('.json'):
        file_path = os.path.join(TestSem.JSON, filename)
        
        # Open and load each JSON file
        with open(file_path, 'r') as file:
            data = json.load(file)
            new = {
                'SEMImageID': data['SEMImageID'],
                'ImageOriginalName': data['ImageOriginalName'],
                'ImageHashName': data['ImageHashName'],
                'ImageAVGHashName': data['ImageAVGHashName'],
                'AP_WD': data['AP_WD'],
                'AP_ALIAS_WD': data['AP_ALIAS_WD'],
                'AP_FREE_WD': data['AP_FREE_WD'],
                'AP_BEAM_TIME': data['AP_BEAM_TIME'],
                'AP_STAGE_LOW_Z': data['AP_STAGE_LOW_Z'],
                'AP_STAGE_CENTRE_X': data['AP_STAGE_CENTRE_X'],
                'AP_C3': data['AP_C3'],
                'AP_EHT_CAL_30KV': data['AP_EHT_CAL_30KV'],
                'AP_EHT_CAL_10KV': data['AP_EHT_CAL_10KV'],
                'AP_EHT_CAL_100': data['AP_EHT_CAL_100'],
                'AP_EHT_CAL_24KV': data['AP_EHT_CAL_24KV'],
                'AP_MINMAG': data['AP_MINMAG'],
            }
            all_data.append(new)

print(f"Loaded {len(all_data)} files.")

In [ ]:
all_data[0]

In [ ]:
for dicts in all_data:
    if dicts['SEMImageID'] == 111:
        mydict = dicts
        break

mydict

# Metadata Extraction

In [ ]:
"""
Load SEM metadata JSON files and extract the 12 settings fields as
clean numeric values (+ units), instead of raw 'Label = value unit' strings.
"""

# Matches: "<label> = <number> <unit>"  e.g. "WD = 3.198 mm"
_NUM_PATTERN = re.compile(r"=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*([^\s=]*)\s*$")


def parse_value(raw_string):
    """
    Turn 'WD = 3.198 mm' into (3.198, 'mm').
    Returns (None, None) if the string can't be parsed (e.g. field is None
    or empty in the source JSON).
    """
    if not raw_string:
        return None, None
    match = _NUM_PATTERN.search(raw_string)
    if not match:
        return None, None
    return float(match.group(1)), match.group(2)


def load_all_metadata(folder_path=TestSem.JSON):
    all_rows = []

    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r") as f:
            data = json.load(f)

        row = {
            "SEMImageID": data["SEMImageID"],
            "ImageOriginalName": data["ImageOriginalName"],
            "ImageHashName": data["ImageHashName"],
            "ImageAVGHashName": data["ImageAVGHashName"],  # bug fix: was hardcoded before
        }

        for key in TestSem.SETTINGS_KEYS:
            value, unit = parse_value(data.get(key))
            row[key] = value # numeric column, e.g. AP_WD = 3.198
            row[f"{key}_unit"] = unit # e.g. AP_WD_unit = "mm"

        all_rows.append(row)

    df = pd.DataFrame(all_rows)
    print(f"Loaded {len(df)} files.")
    return df


df = load_all_metadata()
print(df[["SEMImageID"] + TestSem.SETTINGS_KEYS].head())
print()
print("Null counts per settings field (check for parsing failures or missing data):")
print(df[TestSem.SETTINGS_KEYS].isnull().sum())

In [ ]:
sorted_df = df.sort_values(by='SEMImageID', ascending=True).reset_index(drop=True)
sorted_df.head()

# Crop Rendering

In [ ]:
# Choose whatever crop you want
crop_path = "../data/SEM/crops/107823_5bd1219683c1c668d1f589edc20c1feec9ca7699bafd0bd6ba331e46cb0540cd_512.npy"

# Load crop
crop = np.load(crop_path)

print(f"Shape: {crop.shape}")
print(f"Dtype: {crop.dtype}")
print(f"Min: {crop.min()}, Max: {crop.max()}")

In [ ]:
# Render
plt.figure(figsize=(8, 8))
plt.imshow(crop, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# Convert to a displayable 8-bit image
crop_min = crop.min()
crop_max = crop.max()

if crop_max > crop_min:
    crop_uint8 = ((crop - crop_min) / (crop_max - crop_min) * 255).astype(np.uint8)
else:
    crop_uint8 = np.zeros_like(crop, dtype=np.uint8)

output_path = "/Users/giacomo/Desktop/retrieval_query.png"
Image.fromarray(crop_uint8).save(output_path)

display(FileLink(output_path))